# DataevalFeasibility Analytics Store Demo

This notebook demonstrates the end-to-end workflow for the DataevalFeasibility capability with the analytics store:

1. Load a dataset
2. Run the feasibility capability (Bayes Error Rate estimation)
3. Extract scalar records
4. Write to the analytics store
5. Query results via SQL

## Setup

In [ ]:
import tempfile
from pathlib import Path

from checkmaite.core.analytics_store import AnalyticsStore, ParquetBackend
from checkmaite.core.object_detection import DataevalFeasibility
from checkmaite.core.object_detection.dataset_loaders import CocoDetectionDataset

## Load Dataset

We use the small COCO test dataset included in the repository.

In [ ]:
data_root = Path("../../..")
coco_dir = data_root / "tests" / "data_for_tests" / "coco_resized_val2017"

dataset = CocoDetectionDataset(
    root=str(coco_dir),
    ann_file=str(coco_dir / "instances_val2017_resized_6.json"),
)

print(f"Dataset ID: {dataset.metadata['id']}")
print(f"Images: {len(dataset)}")
print(f"Classes: {dataset.metadata.get('index2label', {})}")

## Run DataevalFeasibility Capability

The feasibility capability estimates the Bayes Error Rate (BER) via kNN classification
on embeddings of ground-truth instance crops. The BER measures how well a kNN classifier
can distinguish object classes from their visual appearance alone.

- **Low BER**: classes are well-separated, problem is feasible
- **High BER**: class ambiguity or label noise, problem may not be feasible

The OD variant also computes dataset health statistics (small objects, truncated
bounding boxes, overlapping detections).

In [ ]:
capability = DataevalFeasibility()
run = capability.run(datasets=[dataset], use_cache=False)

print(f"Run UID: {run.run_uid[:24]}...")
print(f"\nBER upper bound: {run.outputs.ber_upper:.4f}")
print(f"BER lower bound: {run.outputs.ber_lower:.4f}")
print(f"Num instances: {run.outputs.num_instances}")
print(f"Num classes: {run.outputs.num_classes}")
print(f"\nHealth stats:")
print(f"  Small object ratio: {run.outputs.health_stats.small_object_ratio:.2%}")
print(f"  Truncated bbox ratio: {run.outputs.health_stats.truncated_bbox_ratio:.2%}")
print(f"  Overlap image ratio: {run.outputs.health_stats.overlap_image_ratio:.2%}")
if run.outputs.health_stats.warnings:
    print(f"  Warnings: {run.outputs.health_stats.warnings}")

## Extract Records

The `extract()` method distills the feasibility outputs into flat scalar records
suitable for the analytics store. The OD variant includes health statistics
alongside BER bounds.

In [ ]:
records = run.extract()
print(f"Extracted {len(records)} record(s)\n")

record = records[0]
print(f"Table: {record.table_name}")
print(f"Dataset ID: {record.dataset_id}")
print(f"\n--- BER Bounds ---")
print(f"  BER upper: {record.ber_upper:.4f}")
print(f"  BER lower: {record.ber_lower:.4f}")
print(f"\n--- OD Health Stats ---")
print(f"  Num instances: {record.num_instances}")
print(f"  Num classes: {record.num_classes}")
print(f"  Small object ratio: {record.small_object_ratio}")
print(f"  Truncated bbox ratio: {record.truncated_bbox_ratio}")
print(f"  Overlap image ratio: {record.overlap_image_ratio}")
print(f"  Health warning count: {record.health_warning_count}")

## Write to Analytics Store

In [ ]:
store_dir = tempfile.mkdtemp(prefix="feasibility_demo_")
store = AnalyticsStore(ParquetBackend(store_dir))

store.write([run])

print(f"Store path: {store_dir}")
print(f"Tables: {store.list_tables()}")
print(f"\nFeasibility table schema: {store.describe_table('dataeval_feasibility')}")

## Query via SQL

All results are now queryable via standard SQL.

In [ ]:
# View all feasibility records
df = store.query_sql("SELECT * FROM dataeval_feasibility")
print(df)

In [ ]:
# Query specific fields
df = store.query_sql("""
    SELECT
        dataset_id,
        ber_upper,
        ber_lower,
        num_instances,
        num_classes,
        small_object_ratio,
        health_warning_count
    FROM dataeval_feasibility
""")
print(df)

In [ ]:
# Check the auto-populated runs table
df_runs = store.query_sql("""
    SELECT run_uid, capability_table, entity_type, entity_id
    FROM runs
""")
print(df_runs)

## Cross-Capability JOIN Example

The `dataset_id` field enables direct JOINs with other capability tables. For example, if you also ran `DataevalBias` or `DataevalCleaning` on the same dataset:

```sql
SELECT
    f.dataset_id,
    f.ber_upper,
    f.ber_lower,
    b.coverage_uncovered_ratio,
    b.balance_mean
FROM dataeval_feasibility f
JOIN dataeval_bias b ON f.dataset_id = b.dataset_id
```

This is possible because both `DataevalFeasibilityRecord` and `DataevalBiasRecord` include `dataset_id` as a shared key.